# Stitch Integration

This notebook demonstrates the **Stitch** stage of the Foreign Whispers dubbing pipeline.
It performs final video assembly: combining the original video with dubbed TTS audio and
rolling two-line translated captions via ffmpeg. The stitch uses audio-only remux
(no re-encoding), preserving original video quality.

**Prerequisites:**
- The Docker stack must be running (`docker compose --profile nvidia up -d`).
- The API should be accessible at `http://localhost:8080`.
- Prior pipeline stages (download, transcribe, translate, TTS) must have completed for the target video.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

video_id = "GYQ5yGV_-Oc"

class _NoopSpan:
    def __enter__(self): return self
    def __exit__(self, *a): pass
class _noop:
    @staticmethod
    def span(name, **kw): return _NoopSpan()
    @staticmethod
    def info(*a, **kw): pass
logfire = _noop()

print("Setup complete")

Project root: /Users/adit/Documents/Coding/AIAssignments/AiProject/foreign-whispers
Setup complete


## Stitch Video

Call the API to combine the original video with the dubbed TTS audio track.
The stitch endpoint replaces the audio stream via ffmpeg remux (no video re-encoding)
and generates rolling two-line VTT captions.

In [2]:
video_id = "GYQ5yGV_-Oc"  # replace with your video ID

with logfire.span("stitch", video_id=video_id):
    result = fw.stitch(video_id)

print(f"Video ID:   {result['video_id']}")
print(f"Video path: {result['video_path']}")
print(f"Config:     {result['config']}")

NameError: name 'fw' is not defined

## Inspect Output Artifacts

The stitch stage produces files in two directories:

- `pipeline_data/api/dubbed_videos/{config}/` — final dubbed MP4 files
- `pipeline_data/api/dubbed_captions/` — target-language VTT caption files

In [3]:
dubbed_videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_videos"
dubbed_captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "dubbed_captions"

print("Dubbed videos:")
for f in sorted(dubbed_videos_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(dubbed_videos_dir)}  ({size_mb:.1f} MB)")

print()
print("Dubbed captions:")
for f in sorted(dubbed_captions_dir.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(dubbed_captions_dir)}  ({size_kb:.1f} KB)")

Dubbed videos:

Dubbed captions:


## View Generated Captions

The stitch stage generates VTT captions in a rolling two-line format:
the current translated line appears on top, and the previous line is shown
on the bottom, giving viewers context continuity.

In [4]:
# Find the VTT file for this video
vtt_files = list(dubbed_captions_dir.rglob(f"{video_id}*.vtt"))
if not vtt_files:
    vtt_files = list(dubbed_captions_dir.rglob("*.vtt"))

if vtt_files:
    vtt_path = vtt_files[0]
    print(f"Caption file: {vtt_path.name}\n")
    lines = vtt_path.read_text().splitlines()
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"\n... ({len(lines) - 30} more lines)")
    print()
    print("Note the rolling two-line pattern: each cue shows the current")
    print("translated line on top and the previous line on the bottom,")
    print("providing continuity for the viewer.")
else:
    print("No VTT files found. Run the stitch step first.")

No VTT files found. Run the stitch step first.


In [9]:
import shutil
from pathlib import Path

# Clear the Coqui TTS cache
tts_cache = Path.home() / ".local" / "share" / "tts"
if tts_cache.exists():
    shutil.rmtree(tts_cache)
    print(f"Cleared TTS cache at {tts_cache}")

# Also try the macOS location
tts_cache2 = Path.home() / "Library" / "Application Support" / "tts"
if tts_cache2.exists():
    shutil.rmtree(tts_cache2)
    print(f"Cleared TTS cache at {tts_cache2}")

print("Cache cleared — rerun the TTS cell to redownload the model")

Cleared TTS cache at /Users/adit/Library/Application Support/tts
Cache cleared — rerun the TTS cell to redownload the model


In [10]:
from api.src.services.tts_engine import text_file_to_speech
from pathlib import Path
import json

# Load and trim segments to first 2 minutes
trans_path = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos" / "GYQ5yGV_-Oc.json"
trans_data = json.loads(trans_path.read_text())

# Keep only segments within first 2 minutes
trimmed = {
    **trans_data,
    "segments": [s for s in trans_data["segments"] if s["start"] < 120]
}

trimmed_path = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos" / "GYQ5yGV_-Oc_2min.json"
trimmed_path.write_text(json.dumps(trimmed, indent=2))
print(f"Trimmed to {len(trimmed['segments'])} segments (first 2 minutes)")

# Run TTS baseline (no alignment)
output_path = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox" / "baseline"
output_path.mkdir(parents=True, exist_ok=True)
print("Running baseline TTS...")
text_file_to_speech(str(trimmed_path), str(output_path), alignment=False)

# Run TTS aligned
aligned_path = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox" / "aligned"
aligned_path.mkdir(parents=True, exist_ok=True)
print("Running aligned TTS...")
text_file_to_speech(str(trimmed_path), str(aligned_path), alignment=True)

Trimmed to 29 segments (first 2 minutes)
Running baseline TTS...
[tts] Chatterbox not available (HTTPConnectionPool(host='localhost', port=8020): Max retries exceeded with url: /v1/audio/speech (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8020): Failed to establish a new connection: [Errno 61] Connection refused"))), falling back to local Coqui
[tts] Using local Coqui TTS on cpu
 > Downloading model to /Users/adit/Library/Application Support/tts/tts_models--es--mai--tacotron2-DDC
 > Model's license - MPL
 > Check https://www.mozilla.org/en-US/MPL/2.0/ for more info.
 > Downloading model to /Users/adit/Library/Application Support/tts/vocoder_models--universal--libri-tts--fullband-melgan
 > Model's license - MPL
 > Check https://www.mozilla.org/en-US/MPL/2.0/ for more info.
 > Using model: Tacotron2
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | 

In [11]:
# Run TTS aligned
aligned_path = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox" / "aligned"
aligned_path.mkdir(parents=True, exist_ok=True)
print("Running aligned TTS...")
text_file_to_speech(str(trimmed_path), str(aligned_path), alignment=True)
print("Done!")

Running aligned TTS...
generating GYQ5yGV_-Oc_2min.wav...[tts] EN transcript not found at /Users/adit/Documents/Coding/AIAssignments/AiProject/foreign-whispers/pipeline_data/api/transcriptions/whisper/GYQ5yGV_-Oc_2min.json, alignment skipped
 > Text splitted to sentences.
['60 minutos con el tiempo.']
 > Text splitted to sentences.
['¿Cuál es el peor escenario que te preocupa?']
 > Text splitted to sentences.
['Es que está cerrado durante semanas y semanas y semanas aquí y empiezas a ver el mundo']
[tts] TTS failed for segment (stack expects each tensor to be equal size, but got [1, 29] at entry 0 and [1, 41] at entry 1), using silence
 > Text splitted to sentences.
['la economía realmente impactada porque es la sangre de la vida en cierta medida.']
[tts] TTS failed for segment (stack expects each tensor to be equal size, but got [1, 41] at entry 0 and [1, 83] at entry 49), using silence
 > Text splitted to sentences.
['Así que la realidad es lo más largo que esto sucede, el mayor impa

In [ ]:
import librosa

for mode in ["baseline", "aligned"]:
    wav = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox" / mode / "GYQ5yGV_-Oc_2min.wav"
    y, sr = librosa.load(str(wav), sr=None)
    duration = len(y) / sr
    print(f"{mode}: {duration:.2f}s  (video first 2min = 120s)")

ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.100 / 60. 26.100
  libavcodec     62. 28.100 / 62. 28.100
  libavformat    62. 12.100 / 62. 12.100
  libavdevice    62.  3.100 / 62.  3.100
  libavfilter    11. 14.100 / 11. 14.100
  libswscale      9.  5.100 /  9.  5.100
  libswresample   6.  3.100 /  6.  3.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/adit/Documents/Coding/AIAssignments/AiProject/foreign-whispers/pipeline_data/api/videos/GYQ5yGV_-Oc.mp4':
  Metadata:
    major_brand     : m

✅ baseline: /Users/adit/Documents/Coding/AIAssignments/AiProject/foreign-whispers/pipeline_data/api/dubbed_videos/GYQ5yGV_-Oc_baseline.mp4
✅ aligned: /Users/adit/Documents/Coding/AIAssignments/AiProject/foreign-whispers/pipeline_data/api/dubbed_videos/GYQ5yGV_-Oc_aligned.mp4
Done! Check pipeline_data/api/dubbed_videos/


[out#0/mp4 @ 0x13e1050a0] video:4391KiB audio:283KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 1.448500%
frame= 3639 fps=0.0 q=-1.0 Lsize=    4741KiB time=00:02:01.43 bitrate= 319.8kbits/s speed= 388x elapsed=0:00:00.31    
[aac @ 0x13e108550] Qavg: 49549.781


## Playback

To play the dubbed output:

1. **Frontend:** Open <http://localhost:8501> and select the video from the list.
   The UI will load the dubbed MP4 with captions overlay.

2. **Direct file:** Play the MP4 directly from
   `pipeline_data/api/dubbed_videos/{config}/{video_id}.mp4` using any media player
   (e.g., VLC, mpv). Load the corresponding VTT file from `pipeline_data/api/dubbed_captions/`
   as an external subtitle track.

## Summary

The stitch stage produced:

- **Dubbed MP4** in `pipeline_data/api/dubbed_videos/{config}/` — original video with replaced audio track
- **VTT captions** in `pipeline_data/api/dubbed_captions/` — rolling two-line translated subtitles

Audio-only remux means no quality loss on the video track: ffmpeg copies the video
stream as-is and only replaces the audio stream with the synthesized TTS output.